# Audit des données

Objectif : vérifier la disponibilité et la qualité des données
avant de comparer les portefeuilles.

Première étape : collecte pilote de SPY, IEF, GLD et BIL
sur la période 2019–2023.

In [3]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf
from IPython.display import display

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from config import TICKERS, PILOT_START, PILOT_DOWNLOAD_END

# on vérifie l'environnement et les ETF sélectionnés
print("Projet :", project_dir)
print("Python :", sys.executable)
print("Version pandas :", pd.__version__)
print("Version yfinance :", yf.__version__)
print("ETF :", TICKERS)

Projet : c:\Users\DHERALE\Desktop\Projet-Python-ENSAE
Python : C:\Users\DHERALE\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
Version pandas : 2.2.3
Version yfinance : 1.7.0
ETF : ['SPY', 'IEF', 'GLD', 'BIL']


In [4]:
# on commence un mois avant le début de la période étudiée
download_start = (
    pd.Timestamp(PILOT_START) - pd.DateOffset(months=1)
).strftime("%Y-%m-%d")

# on télécharge les donées SPY journalières
spy_test = yf.download(
    "SPY",
    start=download_start,
    end=PILOT_DOWNLOAD_END,
    interval="1d",
    auto_adjust=False,
    actions=True,
    keepna=True,
    multi_level_index=False,
    progress=False,
)

# Vérifier que le téléchargement a fonctionné
if spy_test.empty:
    raise ValueError("Aucune donnée téléchargée pour SPY.")

print("Dimensions :", spy_test.shape)
print("Première date :", spy_test.index.min())
print("Dernière date :", spy_test.index.max())

display(spy_test.head())

Dimensions : (1277, 9)
Première date : 2018-12-03 00:00:00
Dernière date : 2023-12-29 00:00:00


,Adj Close,Capital Gains,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,,
2018-12-03,247.790207,0.0,279.299988,0.0,280.399994,277.510010,280.279999,0.0,103176300
2018-12-04,239.761246,0.0,270.250000,0.0,278.850006,269.899994,278.369995,0.0,177986000
2018-12-06,239.397522,0.0,269.839996,0.0,269.970001,262.440002,265.920013,0.0,204185400
2018-12-07,233.834808,0.0,263.570007,0.0,271.220001,262.630005,269.459991,0.0,161018900
2018-12-10,234.278366,0.0,264.070007,0.0,265.160004,258.619995,263.369995,0.0,151445900


In [5]:
expected_columns = ["Close", "Adj Close", "Dividends", "Stock Splits"]

# on vérifie la présence de chaque colonne 
for column in expected_columns:
    if column not in spy_test.columns:
        raise ValueError(f"Colonne absente : {column}")

display(spy_test[expected_columns].head())

# on compte les valeurs manquantes dans chaque colonne
missing_values = spy_test[expected_columns].isna().sum()
display(missing_values)

,Close,Adj Close,Dividends,Stock Splits
Date,,,,
2018-12-03,279.299988,247.790207,0.0,0.0
2018-12-04,270.250000,239.761246,0.0,0.0
2018-12-06,269.839996,239.397522,0.0,0.0
2018-12-07,263.570007,233.834808,0.0,0.0
2018-12-10,264.070007,234.278366,0.0,0.0


Close           0
Adj Close       0
Dividends       0
Stock Splits    0
dtype: int64

In [6]:
distributions = spy_test.loc[
    spy_test["Dividends"] > 0,
    ["Close", "Adj Close", "Dividends"]
]

print("Nombre de distributions observées :", len(distributions))
display(distributions.head())

Nombre de distributions observées : 21


,Close,Adj Close,Dividends
Date,,,
2018-12-21,240.699997,214.791977,1.435
2019-03-15,281.309998,252.136673,1.233
2019-06-21,294.000000,264.792145,1.432
2019-09-20,298.279999,269.887634,1.384
2019-12-20,320.730011,291.627441,1.570
